In [ ]:
import os
import datetime
import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pdc
import seaborn as sns
import tensorflow as tf
from keras.layers import Dropout
import random
from tensorflow.keras.models import save_model
from tensorflow.keras.models import load_model
from sklearn import preprocessing
from scipy.signal import stft
from scipy.signal import spectrogram
import pickle
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from numpy.random import seed

In [ ]:
# Set seeds
# np
seed(3)
# tf
tf.random.set_seed(3)

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

standard_avg = np.load('/content/drive/My Drive/S_D_ERP_RNN_2/avg_PRECED_S_array_meg_signif_ch.npy', allow_pickle=True)
deviant_avg = np.load('/content/drive/My Drive/S_D_ERP_RNN_2/avg_D_array_meg_signif_ch.npy', allow_pickle=True)

# Get average over the 6 channels
standard_avg = np.mean(standard_avg, axis = 0)
deviant_avg = np.mean(deviant_avg, axis = 0)

# Convert to fT
standard_avg = standard_avg*1e15
deviant_avg = deviant_avg*1e15

In [ ]:
# Read empirically-derived AEF data
PRECED_S_meg_signif_ch = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/Controls_PRECED_S_indiv_meg_signif_ch_max_ampl.npy', allow_pickle=True)
Patient_PRECED_S_meg_signif_ch = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/Patients_PRECED_S_indiv_meg_signif_ch_max_ampl.npy', allow_pickle=True)

D_meg_signif_ch = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/Controls_D_indiv_meg_signif_ch_max_ampl.npy', allow_pickle=True)
Patient_D_meg_signif_ch = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/Patients_D_indiv_meg_signif_ch_max_ampl.npy', allow_pickle=True)

patients_S_ch_sig_control = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/avg_PRECED_S_array_meg_signif_ch_ctrl_signif_pat.npy', allow_pickle=True)
patients_D_ch_sig_control = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/avg_D_array_meg_signif_ch_ctrl_signif_pat.npy', allow_pickle=True)

patients_S_ch_sig_control = patients_S_ch_sig_control*1e15
patients_D_ch_sig_control = patients_D_ch_sig_control*1e15

# Add noise after standardisation.

# Add noise
def gaussian_noise(x,mu,std):
    noise = np.random.normal(mu, std, size = x.shape)
    x_noisy = x + noise
    return x_noisy

mu=0.0
std = 0.6

# Make arrays for no_samples x timepoints
noisy_S = np.zeros([1200, 138], dtype = 'object')
noisy_D = np.zeros([1200, 138], dtype = 'object')

standard_avg_post_s = stand_test_stan
deviant_avg_post_s = stand_test_dev

for i in range(1000):
    noisy_S_s = gaussian_noise(standard_avg_post_s,mu,std)
    noisy_S[i, :] = noisy_S_s
    noisy_D_s = gaussian_noise(deviant_avg_post_s,mu,std)
    noisy_D[i, :] = noisy_D_s

In [ ]:
### Generate inputs


# Standard inputs

freqs = [250, 500, 1000]

S_list = []

for i in range(len(freqs)):

  fs = 5000 # Sampling rate in Hz
  dt = 1/fs # Time resolution in seconds
  t = np.arange(0, 0.05*3+0.398, dt) # Time vector 0.398/2 = 0.199

  # Generate the sinewave signal
  f = freqs[i]
  sine1 = np.sin(2*np.pi*f*t[:int(0.05*fs)])
  space1 = np.zeros(int(0.199*fs))
  sine2 = np.sin(2*np.pi*f*t[int(0.05*fs+0.199*fs):int(0.1*fs+0.199*fs)])
  space2 = np.zeros(int(0.199*fs))
  sine3 = np.sin(2*np.pi*f*t[int(0.1*fs+0.398*fs):int(0.1*fs+0.398*fs+0.05*fs)])

  x = np.concatenate([sine1, space1, sine2, space2, sine3])

  # Calculate the STFT
  nperseg = int(0.025*fs) # Window size (i.e., 25ms)
  noverlap = int(0.021*fs) # Overlap size (i.e., 21ms)
  f, t, Zxx = stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap)

  # Plot the spectrogram
  plt.pcolormesh(t*1000, f, np.abs(Zxx), vmin=0, vmax=np.max(np.abs(Zxx)))
  plt.xlabel('Time (ms)', fontsize = 17)
  plt.ylabel('Frequency (Hz)', fontsize = 17)
  plt.xticks(fontsize=16)
  plt.yticks(fontsize=16)
  plt.ylim(0, 1200)
  plt.yticks(np.arange(0, 1250, 250))
  plt.colorbar()
  plt.tight_layout()
#   plt.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/input_{freqs[i]}Hz.pdf')
  plt.show()

  input_S = np.zeros([400, noisy_S.shape[1], Zxx.shape[0]], dtype = 'object') # keep name w _1000

  # Get abs value of Zxx:
  Zxx_abs = np.abs(Zxx)

  for j in range(400):
    input_S[j, :, :] = Zxx_abs.T

  S_list.append(input_S)

# all types of S in input_S
input_S = np.zeros([1200, 138, 63], dtype = 'object')
input_S[:400] = S_list[0]
input_S[400:800] = S_list[1]
input_S[800:1200] = S_list[2]


# Deviants inputs

# formula: N!/(N-k)! = 6 (in this case: 3 freqs; 2 at a time: first 2 equal; third = distinct)
# 250x2 500; 250x2 1000; 500x2 1000; 500x2 250; 1000x2 250; 1000x2 500. -> Make 200 of each type!

import itertools

freqs = [250, 500, 1000]
S_D_pairs = []

# Create the tournament fixture list
for permutation in itertools.permutations(freqs, 2):
    S = permutation[0]
    D = permutation[1]
    S_D_pairs.append((S, D))

Dev_list = []

for i in range(len(S_D_pairs)):
  fs = 5000 # Sampling rate in Hz
  dt = 1/fs # Time resolution in seconds
  t = np.arange(0, 0.05*3+0.398, dt) # Time vector 0.398/2 = 0.199
  f = S_D_pairs[i][0]
  fD = S_D_pairs[i][1]

  # Generate the sinewave signal
  sine1 = np.sin(2*np.pi*f*t[:int(0.05*fs)])
  space1 = np.zeros(int(0.199*fs))
  sine2 = np.sin(2*np.pi*f*t[int(0.05*fs+0.199*fs):int(0.1*fs+0.199*fs)])
  space2 = np.zeros(int(0.199*fs))
  sine3 = np.sin(2*np.pi*fD*t[int(0.1*fD+0.398*fD):int(0.1*fD+0.398*fD+0.05*fs)])

  x = np.concatenate([sine1, space1, sine2, space2, sine3])


  # Calculate the STFT
  nperseg = int(0.025*fs) # Window size (i.e., 25ms)
  noverlap = int(0.021*fs) # Overlap size (i.e., 21ms)
  f, t, Zxx = stft(x, fs=fs, nperseg=nperseg, noverlap=noverlap) # each Zxx has shape (63, 138)

  # Plot the spectrogram
  plt.pcolormesh(t*1000, f, np.abs(Zxx), vmin=0, vmax=np.max(np.abs(Zxx)))
  plt.xlabel('Time (ms)', fontsize = 17)
  plt.ylabel('Frequency (Hz)', fontsize = 17)
  plt.xticks(fontsize=16)
  plt.yticks(fontsize=16)
  plt.ylim(0, 1200)
  plt.yticks(np.arange(0, 1250, 250))
  plt.colorbar()
  plt.tight_layout()
#   plt.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/input_{S_D_pairs[i][0]}_{S_D_pairs[i][1]}Hz.pdf')
  plt.show()

  input_D = np.zeros([200, noisy_D.shape[1], Zxx.shape[0]], dtype = 'object')

  # Get abs value of Zxx:
  Zxx_abs = np.abs(Zxx)

  for j in range(200):
    input_D[j, :, :] = Zxx_abs.T

  Dev_list.append(input_D)

# All types of D in input_D
input_D = np.zeros([1200, 138, 63], dtype = 'object')
input_D[:200] = Dev_list[0]
input_D[200:400] = Dev_list[1]
input_D[400:600] = Dev_list[2]
input_D[600:800] = Dev_list[3]
input_D[800:1000] = Dev_list[4]
input_D[1000:1200] = Dev_list[5]

In [ ]:
noisy_S = noisy_S.reshape(noisy_S.shape[0], noisy_S.shape[1], 1)
noisy_D = noisy_D.reshape(noisy_D.shape[0], noisy_D.shape[1], 1)

# Shuffle arrays before splitting -> careful: shuffle inputs and labels in the same way!
from sklearn.utils import shuffle
input_S, noisy_S = shuffle(input_S, noisy_S, random_state=0)
input_D, noisy_D = shuffle(input_D, noisy_D, random_state=1)

# train data: 70%
input_S_train = input_S[0:int(0.7*input_S.shape[0])]
labels_S_train = noisy_S[0:int(0.7*noisy_S.shape[0])]
input_D_train = input_D[0:int(0.7*input_D.shape[0])]
labels_D_train = noisy_D[0:int(0.7*noisy_D.shape[0])]

# Validation data: 15%
input_S_valid = input_S[int(0.7*input_S.shape[0]):int(0.7*input_S.shape[0])+int(0.15*input_S.shape[0])]
labels_S_valid = noisy_S[int(0.7*noisy_S.shape[0]):int(0.7*noisy_S.shape[0])+int(0.15*noisy_S.shape[0])]
input_D_valid = input_D[int(0.7*input_D.shape[0]):int(0.7*input_D.shape[0])+int(0.15*input_D.shape[0])]
labels_D_valid = noisy_D[int(0.7*noisy_D.shape[0]):int(0.7*noisy_D.shape[0])+int(0.15*noisy_D.shape[0])]

# Test data: 15%. Now substract from total no. of samples.
input_S_test = input_S[int(0.7*input_S.shape[0])+int(0.15*input_S.shape[0]):]
labels_S_test = noisy_S[int(0.7*noisy_S.shape[0])+int(0.15*noisy_S.shape[0]):]
input_D_test = input_D[int(0.7*input_D.shape[0])+int(0.15*input_D.shape[0]):]
labels_D_test = noisy_D[int(0.7*noisy_D.shape[0])+int(0.15*noisy_D.shape[0]):]

train_array = np.concatenate([input_S_train, input_D_train])
labels_train_array = np.concatenate([labels_S_train, labels_D_train])

valid_array = np.concatenate([input_S_valid, input_D_valid])
labels_valid_array = np.concatenate([labels_S_valid, labels_D_valid])

test_array = np.concatenate([input_S_test, input_D_test])
labels_test_array = np.concatenate([labels_S_test, labels_D_test])

# Reshape labels arrays so that they have shape: no_trials x no_timepoints (without the 3rd dimension which would be 1 anyways)
labels_train_array = labels_train_array.reshape(train_array.shape[0], train_array.shape[1])
labels_valid_array = labels_valid_array.reshape(valid_array.shape[0], valid_array.shape[1])
labels_test_array = labels_test_array.reshape(test_array.shape[0], test_array.shape[1])

# Create combined tf dataset with inputs and labels

train_array = np.array(train_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/train_array.npy', train_array)
labels_train_array = np.array(labels_train_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/labels_train_array.npy', labels_train_array)
valid_array = np.array(valid_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/valid_array.npy', valid_array)
labels_valid_array = np.array(labels_valid_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/labels_valid_array.npy', labels_valid_array)
test_array = np.array(test_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/test_array.npy', test_array)
labels_test_array = np.array(labels_test_array, dtype='float64')
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data/labels_test_array.npy', labels_test_array)

# Create tf datasets
train_tf_dataset = tf.data.Dataset.from_tensor_slices((train_array, labels_train_array)) # train
valid_tf_dataset = tf.data.Dataset.from_tensor_slices((valid_array, labels_valid_array)) # validation
test_tf_dataset = tf.data.Dataset.from_tensor_slices((test_array, labels_test_array)) # test

# Batch tfds dataset - batch size = 32
train_tf_dataset = train_tf_dataset.batch(128)
valid_tf_dataset = valid_tf_dataset.batch(128)
test_tf_dataset = test_tf_dataset.batch(128)

# Extract example batch from tfds
example_batch = next(iter(train_tf_dataset))# Build the RNN model -> SimpleRNN -> fully connected RNN where output -> fed back to input



In [ ]:
# Build the RNN model -> SimpleRNN -> fully connected RNN where output -> fed back to input

simple_rnn_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(64, return_sequences=True, activation = 'relu'), # recurrent layer with 64 units
    tf.keras.layers.SimpleRNN(64, return_sequences=True, activation = 'relu'), # recurrent layer with 64 units
    tf.keras.layers.SimpleRNN(64, return_sequences=True, activation = 'relu'), # recurrent layer with 64 units
    tf.keras.layers.SimpleRNN(64, return_sequences=True, activation = 'relu'), # recurrent layer with 64 units
    tf.keras.layers.SimpleRNN(1, return_sequences=True, activation = 'linear')
])

# Test that model can successfully process inputs
print('Input shape:', example_batch[0].shape) # batchsize x timesteps x features
print('Output shape:', simple_rnn_model(example_batch[0]).shape)


In [ ]:
# Run model

# Data from CONTROLS is used to train the RNN

MAX_EPOCHS = 100

def compile_and_fit_class(model, train_data, valid_data, patience = 3):

  # Save model after each epoch
  checkpoint_filepath = '/content/drive/My Drive/S_D_ERP_RNN_final/data_final/model.{epoch:02d}-{val_loss:.4f}.h5'

  model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
  filepath=checkpoint_filepath,
  save_weights_only=False,
  monitor='val_loss',
  mode='min',
  save_best_only=False,
  save_freq='epoch',
  verbose=1)

  # patience: magnitude change parameter -> min_delta = x; an absolute change of less than min_delta, will count as no improvement
  # Stop training after x (patience) epochs in which loss stops decreasing.
  early_stopping = tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = patience, mode = 'min')

  # make a list with my_callbacks
  my_callbacks = [early_stopping, model_checkpoint_callback]

  # Compile
  # make learning rate even smaller (default = 0.001)
  optimiser = tf.keras.optimizers.Adam(learning_rate=0.00025)
  model.compile(optimizer = optimiser, loss = 'mean_squared_error', metrics = ['mean_squared_error'])

  # fit model
  history = model.fit(train_data, epochs = MAX_EPOCHS, validation_data = valid_data, callbacks=[my_callbacks])

  return history

# Apply the function
history = compile_and_fit_class(simple_rnn_model, train_data = train_tf_dataset, valid_data = valid_tf_dataset)

# Save the trained model in a h5 file
save_model(simple_rnn_model, "/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5")

# save history -> this is the training & validation history of the original model
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/history_training_val.npy', history.history)



In [ ]:
# Plot loss (mean squared error)

# load history of original model
history_training_val=np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/history_training_val.npy',allow_pickle='TRUE').item() #history = dictionary --> read from google drive!

# Plot loss
loss_train = history_training_val['loss']
loss_val = history_training_val['val_loss']

epochs = range(1, 11)
plt.figure(figsize=(6, 3))
#plt.figure(figsize=(4, 4))
plt.plot(epochs, loss_train, 'g', label='Training loss')
plt.plot(epochs, loss_val, 'b', label='Validation loss')
#plt.title('Training and Validation loss (MSE)')
plt.xlabel('Epochs', fontsize = 14)
plt.ylabel('Loss (MSE)', fontsize = 14)
plt.legend()
plt.tight_layout()
# save plot
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/training_valid_loss_orig.pdf')
plt.show()

In [ ]:
# Evaluate the model

# test the model
simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

score=simple_rnn_model_orig.evaluate(test_tf_dataset, verbose=2)

print(score)

In [ ]:
## Use the trained RNN to predict the S and D from the test data. Average across frequencies for each of the S and D type.

## Plot prediction by the output layer of the RNN.

output_list_S_250_orig_model = []
output_list_S_500_orig_model = []
output_list_S_1000_orig_model = []

output_list_S_orig_model_array = np.zeros([3, 138], dtype='object')

output_list_D_250_1000_orig_model = []
output_list_D_500_1000_orig_model = []
output_list_D_250_500_orig_model = [] 

output_list_D_orig_model_array = np.zeros([3, 138], dtype='object')


simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

t = np.arange(0, 137*4+1, 4)

for i in range(len(simple_rnn_model_orig.layers)):
    if i == 4: # output layer
      layer = simple_rnn_model_orig.layers[i]
      if isinstance(layer, tf.keras.layers.SimpleRNN):
          output = layer.output
          activation_model = tf.keras.models.Model(inputs=simple_rnn_model_orig.input, outputs=output)
          activations_S1 = activation_model.predict(test_array[1].reshape(1, 138, 63)) # 250 S
          activations_S2 = activation_model.predict(test_array[5].reshape(1, 138, 63)) # 500 S
          activations_S3 = activation_model.predict(test_array[10].reshape(1, 138, 63)) # 1000S

          output_list_S_250_orig_model.append(activations_S1)
          output_list_S_500_orig_model.append(activations_S2)
          output_list_S_1000_orig_model.append(activations_S3)

          activations_D1 = activation_model.predict(test_array[297].reshape(1, 138, 63)) #250 - 1000 D
          activations_D2 = activation_model.predict(test_array[267].reshape(1, 138, 63)) #500 - 1000 D
          activations_D3 = activation_model.predict(test_array[295].reshape(1, 138, 63)) #500 - 1000 D

          output_list_D_250_1000_orig_model.append(activations_D1)
          output_list_D_500_1000_orig_model.append(activations_D2)
          output_list_D_250_500_orig_model.append(activations_D3)

output_list_S_orig_model_array[0, :] = np.array(output_list_S_250_orig_model)[0, 0, :, 0]
output_list_S_orig_model_array[1, :] = np.array(output_list_S_500_orig_model)[0, 0, :, 0]
output_list_S_orig_model_array[2, :] = np.array(output_list_S_1000_orig_model)[0, 0, :, 0]

output_list_D_orig_model_array[0, :] = np.array(output_list_D_250_1000_orig_model)[0, 0, :, 0]
output_list_D_orig_model_array[1, :] = np.array(output_list_D_500_1000_orig_model)[0, 0, :, 0]
output_list_D_orig_model_array[2, :] = np.array(output_list_D_250_500_orig_model)[0, 0, :, 0]

avg_output_S_orig_model = np.mean(output_list_S_orig_model_array, axis = 0)
avg_output_D_orig_model = np.mean(output_list_D_orig_model_array, axis = 0)

plt.figure(figsize=(6, 3))
plt.title('RNN Prediction')
plt.xlabel('Time (ms)')
plt.ylabel('Output Layer Activation')
plt.plot(t, avg_output_S_orig_model, label = 'Standard', color = 'blue', linestyle = "-")
plt.plot(t, avg_output_D_orig_model, label = 'Deviant', color = 'red', linestyle = "-")
plt.legend()
#plt.ylim([-5.5, 6.5])
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig_L_5_prediction_S+D.pdf')


In [ ]:
""" Model prediction vs target (label) """

fig, axs = plt.subplots(2)

# Load original model
simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')
avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

print(test_array.shape, labels_test_array.shape)

t = np.arange(0, 137*4+1, 4)

label_ex1 = labels_test_array[1] # standard
label_ex2 = labels_test_array[297] # deviant

axs[0].plot(t, avg_output_S_orig_model, label = "RNN Prediction", color = 'cornflowerblue')
axs[0].plot(t, label_ex1, label = "Model Target", color = 'darkblue', linestyle = ':')
axs[0].legend(fontsize = 9)
axs[0].set_ylim([-3, 3])
axs[1].plot(t, avg_output_D_orig_model, label = "RNN Prediction", color = 'turquoise')
axs[1].plot(t, label_ex2, label = "Model Target", color = 'darkcyan', linestyle = ':')
axs[1].legend(fontsize = 9)
axs[1].set_ylim([-3, 3])
axs[0].set_title('Standard Evoked Response')
axs[0].set(xlabel='Time (ms)', ylabel='fT')
axs[1].set_title('Deviant Evoked Response')
axs[1].set(xlabel='Time (ms)', ylabel='fT')
fig.tight_layout()
#fig.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/orig_model_RNN_Pred_vs_Target_S+D.pdf')


In [ ]:
""" Plot activations through the hidden layers for S & D inputs"""

def plot_layer_activations(model, test_array, stim_type='Standard'):
    """Plot layer activations for either Standard or Deviant stimuli.
    
    Args:
        model: The trained RNN model
        test_array: Array containing test data
        stim_type: Either 'Standard' or 'Deviant'
    """
    color_bar_limits = [(0, 0.5), (0, 0.5), (0, 0.5), (0, 0.5)]
    fig, axs = plt.subplots(4, figsize=(15, 7))
    hidden_layers_activ = np.zeros([4, 138, 64], dtype='float')

    # Define indices based on stimulus type
    if stim_type == 'Standard':
        indices = [1, 5, 10]  # indices for 250Hz, 500Hz, 1000Hz Standard
    else:  # Deviant
        indices = [297, 267, 295]  # indices for different Deviant combinations

    for i in range(len(model.layers)-1):  # plot activation for layers 1-4
        layer = model.layers[i]
        output_list = np.zeros([3, 138, 64], dtype='float')

        if isinstance(layer, tf.keras.layers.SimpleRNN):
            output = layer.output
            activation_model = tf.keras.models.Model(inputs=model.input, outputs=output)
            
            # Get activations for all three variants
            for j, idx in enumerate(indices):
                activation = activation_model.predict(test_array[idx].reshape(1, 138, 63))
                output_list[j, :, :] = activation[0, :, :]

        # Average activations across variants
        avg_output = np.mean(output_list, axis=0)
        avg_output = avg_output.reshape(138, 64)
        hidden_layers_activ[i] = avg_output

        # Plot heatmap
        avg_output_T = avg_output.T
        heatmap = axs[i].matshow(avg_output_T, cmap='viridis')
        axs[i].set_ylabel(f'Layer {i+1} Units', fontsize=10)
        axs[i].xaxis.set_ticks_position('bottom')
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        heatmap.set_clim(color_bar_limits[i])
        plt.colorbar(heatmap)

    fig.suptitle(stim_type, fontsize=15)
    fig.tight_layout()
    
    # Save figure and data if needed
    # fig.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/orig_model_layer_activations_L1-L4_{stim_type[0]}.pdf')
    # np.save(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/hidden_layers_activ_{stim_type[0]}_orig_model.npy', hidden_layers_activ)
    
    return hidden_layers_activ

# Load original model
simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

# Generate plots for both Standard and Deviant
hidden_layers_activ_S = plot_layer_activations(simple_rnn_model_orig, test_array, 'Standard')
hidden_layers_activ_D = plot_layer_activations(simple_rnn_model_orig, test_array, 'Deviant')


In [ ]:
# Plot activity in hidden layers & Average over units this time

hidden_layers_activ_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/hidden_layers_activ_S_orig_model.npy')
hidden_layers_activ_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/hidden_layers_activ_D_orig_model.npy')

hidden_layers_activ_S_avg_over_units = np.mean(hidden_layers_activ_S, axis = 2)
hidden_layers_activ_D_avg_over_units = np.mean(hidden_layers_activ_D, axis = 2)

t = np.arange(0, 137*4+1, 4)


for l in range(4):
  plt.figure(figsize=(8, 3))
  plt.plot(t, hidden_layers_activ_S_avg_over_units[l], label = 'Standard', color = 'blue', linestyle = "-")
  plt.plot(t, hidden_layers_activ_D_avg_over_units[l], label = 'Deviant', color = 'red', linestyle = "-")
  plt.title(f"Hidden Layer {l+1}", fontsize = 12)
  plt.legend(bbox_to_anchor=(1.1, 1), loc='upper right', fontsize="14")
  plt.tight_layout()
  plt.gca().spines['top'].set_visible(False)
  plt.gca().spines['right'].set_visible(False)
  plt.ylim(0, 0.1)
  plt.xlabel("Time", fontsize = 18)
  plt.ylabel("Activity", fontsize = 18)
  plt.rcParams['xtick.labelsize'] = 16
  plt.rcParams['ytick.labelsize'] = 16
  plt.show()


In [ ]:
# Get weight distribution / ratio of +/- weights in trained model.
simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')
no_layers = 4
weights_array = np.zeros([4, 4096])
for l in range(no_layers):
  simple_rnn_layer_w = simple_rnn_model_orig.layers[l].get_weights()
  recurrent_weights_l = simple_rnn_layer_w[1] # choose recurrent weights
  recurrent_weights_l_flat = recurrent_weights_l.flatten()
  weights_array[l, :] = recurrent_weights_l_flat
  positive_w = recurrent_weights_l_flat[np.where(recurrent_weights_l_flat>0)]
  strength_pos_w = np.sum(positive_w)
  negative_w = recurrent_weights_l_flat[np.where(recurrent_weights_l_flat<0)]
  strength_neg_w = np.sum(negative_w)
  ratio_pos_neg = (positive_w.shape[0])/(negative_w.shape[0])
  print(f"Strength pos/neg ratio layer {l+1}:", strength_pos_w/np.abs(strength_neg_w))
  

weights_array_flat = weights_array.flatten()

orig_model_neg_w = weights_array_flat[np.where(weights_array_flat<0)]
orig_model_pos_w = weights_array_flat[np.where(weights_array_flat>0)]

pos_weights_total = weights_array_flat[np.where(weights_array_flat>0)]
print("Pos weights:", pos_weights_total.shape)
no_pos_weights_total = pos_weights_total.shape[0]
neg_weights_total = weights_array_flat[np.where(weights_array_flat<0)]
print("Neg weights:", neg_weights_total.shape)
no_neg_weights_total = neg_weights_total.shape[0]

total_ratio_pos_neg = no_pos_weights_total/no_neg_weights_total
print(total_ratio_pos_neg)

  
# Calculate strength of all excitatory weights.
pos_weights_strength = np.sum(pos_weights_total, axis = 0)
neg_weights_strength = np.sum(neg_weights_total, axis = 0)

E_I_strength_ratio = pos_weights_strength/neg_weights_strength
print(np.abs(E_I_strength_ratio))

# Calculate average excitation / synapse
avg_exc_syn = pos_weights_strength/no_pos_weights_total

# Calculate average inhibition / synapse
avg_inh_syn = np.abs(neg_weights_strength)/no_neg_weights_total

synapse_E_I_strength_ratio = avg_exc_syn/avg_inh_syn
print(synapse_E_I_strength_ratio)

### Model perturbations. Alterations of weights in all hidden layers.

#### Experiment 1: Negative weights increase.

In [ ]:
# Relative % changes in weights' absolute value
weight_upd_list = list(np.arange(0.005, 0.04, 0.005))
for el in range(len(weight_upd_list)):
  weight_upd_list[el] = round(weight_upd_list[el], 3)
  

""" Experiment 1: Negative weights increase """
simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

# Perturb negative weights (increase by %...) ---> later: do small variations in the level of perturbations between the hidden layers.
simple_rnn_model_w = []
no_layers = 4
new_weight_array_rnn = np.zeros([len(weight_upd_list), no_layers, 2], dtype = 'object')

for j in range(len(weight_upd_list)):
  # Load original model
  simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

  # Perturb hidden layers - L1-L4
  for l in range(no_layers):
    # Get the weights of a particular layer
    simple_rnn_layer_w = simple_rnn_model_orig.layers[l].get_weights()
    # Perturb recurrent weights
    simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] < 0)] = simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] < 0)] + np.abs(weight_upd_list[j]*simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] < 0)])
    new_weight_array_rnn[j, l, 0] = simple_rnn_layer_w[0] # feedf weight = same
    new_weight_array_rnn[j, l, 1] = simple_rnn_layer_w[1] # recurrent weight = the one above

    # update weights
    simple_rnn_model_orig.layers[l].set_weights([new_weight_array_rnn[j, l, 0], new_weight_array_rnn[j, l, 1], simple_rnn_layer_w[2]])

  simple_rnn_model_w.append(simple_rnn_model_orig) # after the 4th layer has been perturbed, add whole perturbed model to the list.

# Evaluate the models
scores_neg_w_perturb_magn = np.zeros([len(weight_upd_list)], dtype = 'object')

for j in range(len(weight_upd_list)):
  # evaluate the model
  score=simple_rnn_model_w[j].evaluate(test_tf_dataset, verbose=2) # the updated version of the original model
  scores_neg_w_perturb_magn[j] = score

# Save loss and mean sq error for TEST
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/scores_neg_w_perturb_magn.npy', scores_neg_w_perturb_magn)

# Save pertubed models
for j in range(len(weight_upd_list)):
  save_model(simple_rnn_model_w[j], f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_neg_w{weight_upd_list[j]}.h5")


# Plot loss.
scores_neg_w_perturb_magn = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/scores_neg_w_perturb_magn.npy', allow_pickle=True)
# this is an array with 8 lists: in each list, list[0] = loss; list[1] = MSE. they are the same.

data = {f'{round(weight_upd_list[0]*100, 2)}%': scores_neg_w_perturb_magn[0][0], f'{round(weight_upd_list[1]*100, 2)}%': scores_neg_w_perturb_magn[1][0], f'{round(weight_upd_list[2]*100,2)}%': scores_neg_w_perturb_magn[2][0],
        f'{round(weight_upd_list[3]*100, 2)}%': scores_neg_w_perturb_magn[3][0], f'{round(weight_upd_list[4]*100, 2)}%': scores_neg_w_perturb_magn[4][0], f'{round(weight_upd_list[5]*100, 2)}%': scores_neg_w_perturb_magn[5][0],
        f'{round(weight_upd_list[6]*100, 2)}%': scores_neg_w_perturb_magn[6][0], f'{round(weight_upd_list[7]*100, 2)}%': scores_neg_w_perturb_magn[7][0]}

plt.bar(data.keys(), data.values(),  color=['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta'])
plt.tick_params(labelsize=12)
plt.ylabel('loss', fontsize = 14)
plt.xlabel('Relative increase of recurrent negative weights', fontsize = 14)
plt.title('Loss across perturbation levels', fontsize = 18)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/loss_neg_w_perturb.pdf')
plt.show()

In [ ]:
# Calculate average excitatory / inhibition ratio at the synapse for each layer.
for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_neg_w{weight_upd_list[j]}.h5")

  print(f"Perturb level: {weight_upd_list[j]}")
  print(" ")

  for l in range(no_layers):
    simple_rnn_layer_w = perturbed_model.layers[l].get_weights()
    recurrent_weights_l = simple_rnn_layer_w[1] # choose recurrent weights

    weights_array_flat = recurrent_weights_l.flatten()

    pos_weights_total = weights_array_flat[np.where(weights_array_flat>0)]
    no_pos_weights_total = pos_weights_total.shape[0]
    print(f"Total pos weights layer {1+l}:", no_pos_weights_total)
    neg_weights_total = weights_array_flat[np.where(weights_array_flat<0)]
    no_neg_weights_total = neg_weights_total.shape[0]
    print(f"Total neg weights layer {1+l}:", no_neg_weights_total)

    pos_weights_strength = np.sum(pos_weights_total, axis = 0)
    neg_weights_strength = np.sum(neg_weights_total, axis = 0)

    E_I_strength_ratio = pos_weights_strength/neg_weights_strength
    #print(np.abs(E_I_strength_ratio))
    #print(np.abs(E_I_strength_ratio))

    # Calculate average excitation / synapse
    avg_exc_syn = pos_weights_strength/no_pos_weights_total

    # Calculate average inhibition / synapse
    avg_inh_syn = neg_weights_strength/no_neg_weights_total

    synapse_E_I_strength_ratio = np.abs(avg_exc_syn/avg_inh_syn)
    print(f"Synapse E/I ratio layer {1+l}:", synapse_E_I_strength_ratio)
    print(" ")

In [ ]:
""" Run Experiment 1 perturbed model for S and D inputs."""

output_list_S_250 = []
output_list_S_500 = []
output_list_S_1000 = []

output_list_S_array = np.zeros([3, 8, 138], dtype='object')

output_list_D_250_1000 = []
output_list_D_500_1000 = []
output_list_D_250_500 = [] # other combinations do not differ from these.

output_list_D_array = np.zeros([3, 8, 138], dtype='object')

for j in range(len(weight_upd_list)):
    perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_neg_w{weight_upd_list[j]}.h5")
    layer = perturbed_model.layers[4]

    if isinstance(layer, tf.keras.layers.SimpleRNN):
        output = layer.output
        activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)
        activations_S1 = activation_model.predict(test_array[1].reshape(1, 138, 63)) # 250 S
        activations_S2 = activation_model.predict(test_array[5].reshape(1, 138, 63)) # 500 S
        activations_S3 = activation_model.predict(test_array[10].reshape(1, 138, 63)) # 1000S
        output_list_S_250.append(activations_S1)
        output_list_S_500.append(activations_S2)
        output_list_S_1000.append(activations_S3)

        activations_D1 = activation_model.predict(test_array[297].reshape(1, 138, 63)) #250 - 1000 D
        output_list_D_250_1000.append(activations_D1)
        activations_D2 = activation_model.predict(test_array[267].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_500_1000.append(activations_D2)
        activations_D3 = activation_model.predict(test_array[295].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_250_500.append(activations_D3)

output_list_S_array[0, :, :] = np.array(output_list_S_250)[:, 0, :, 0]
output_list_S_array[1, :, :] = np.array(output_list_S_500)[:, 0, :, 0]
output_list_S_array[2, :, :] = np.array(output_list_S_1000)[:, 0, :, 0]

output_list_D_array[0, :, :] = np.array(output_list_D_250_1000)[:, 0, :, 0]
output_list_D_array[1, :, :] = np.array(output_list_D_500_1000)[:, 0, :, 0]
output_list_D_array[2, :, :] = np.array(output_list_D_250_500)[:, 0, :, 0]

print(output_list_S_array.shape)
print(output_list_D_array.shape)

# Take average over all 3 S input types (from test set)
avg_output_S = np.mean(output_list_S_array, axis = 0)
avg_output_D = np.mean(output_list_D_array, axis = 0)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_S.npy', avg_output_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_D.npy', avg_output_D)


# Plot all predicted AEFs on the same plot.
plt.figure(figsize=(9, 5))
colors=['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta']

avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_S.npy', allow_pickle=True)
avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_D.npy', allow_pickle=True)

t = np.arange(0, 137*4+1, 4)

legend = ['0.5%', '1%', '1.5%', '2%', '2.5%', '3%', '3.5%', '4%']

line1, = plt.plot(t, avg_output_S_orig_model, color = 'black', linestyle = "-", label = '0%')
line2, = plt.plot(t, avg_output_D_orig_model, color = 'black', linestyle = "--")
for i in range(8):
  line3, = plt.plot(t, avg_output_S[i], color = colors[i], linestyle = "-", label = legend[i])
  line4, = plt.plot(t, avg_output_D[i], color = colors[i], linestyle = "--")
#plt.legend(handles=[line3, line4], loc='upper left')
plt.legend(loc = 'upper left', title = "Perturbation level")
#plt.legend(handles=[line3, line4], loc='upper left')
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.xlabel('Time (ms)', fontsize = 17)
plt.ylabel('RNN Prediction', fontsize = 17)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_predictions_single_plot.pdf')
plt.show()


In [ ]:
''' 1. Max amplitude ratios & MAE MMMN'''

from sklearn.metrics import mean_absolute_error

# Pre-pert max amplitudes.
avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

pre_pert_peak_S = np.max(avg_output_S_orig_model)
pre_pert_peak_D = np.max(avg_output_D_orig_model)

pre_pert_MAE = mean_absolute_error(avg_output_S_orig_model, avg_output_D_orig_model)
print("pre_pert_MAE:", pre_pert_MAE)

neg_w_perturb_peak_ampl_ratios_S = np.zeros([8,], dtype ='object')
neg_w_perturb_peak_ampl_ratios_D = np.zeros([8,], dtype = 'object')

neg_w_perturb_peak_ampl_S = np.zeros([8,], dtype ='object')
neg_w_perturb_peak_ampl_D = np.zeros([8,], dtype = 'object')

neg_w_perturb_MMN_MAE = np.zeros([8,], dtype ='object')

avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_S.npy', allow_pickle=True)
avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_avg_output_D.npy', allow_pickle=True)

for level in range(8):
  pert_level_peak_S = np.max(avg_output_S[level, :])
  pert_level_peak_D = np.max(avg_output_D[level, :])

  neg_w_perturb_peak_ampl_S[level] = pert_level_peak_S
  neg_w_perturb_peak_ampl_D[level] = pert_level_peak_D

  ratio_S_level = pert_level_peak_S/pre_pert_peak_S
  ratio_D_level = pert_level_peak_D/pre_pert_peak_D

  MMN_MAE = mean_absolute_error(avg_output_S[level, :], avg_output_D[level, :])
  neg_w_perturb_MMN_MAE[level] = MMN_MAE

  neg_w_perturb_peak_ampl_ratios_S[level] = ratio_S_level
  neg_w_perturb_peak_ampl_ratios_D[level] = ratio_D_level

print('Standard ratios:', neg_w_perturb_peak_ampl_ratios_S)
print('Deviant ratios:', neg_w_perturb_peak_ampl_ratios_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_ampl_ratios_S.npy', neg_w_perturb_peak_ampl_ratios_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_ampl_ratios_D.npy', neg_w_perturb_peak_ampl_ratios_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_ampl_S.npy', neg_w_perturb_peak_ampl_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_ampl_D.npy', neg_w_perturb_peak_ampl_D)
print('Peak ampl S:', neg_w_perturb_peak_ampl_S)
print('Peak ampl D:', neg_w_perturb_peak_ampl_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_MMN_MAE.npy', neg_w_perturb_MMN_MAE)
print("MAEs:", neg_w_perturb_MMN_MAE)
print(" ")


''' 2. Peak latency'''
delta_t = 4 # 4ms.

pre_pert_peak_latency_S = (np.argmax(avg_output_S_orig_model)-1)*delta_t # transform in ms. 37th timepoint => 36 intervals => 36 x 4ms = 144ms.
pre_pert_peak_latency_D = (np.argmax(avg_output_D_orig_model)-1)*delta_t

print('Pre-perturbation Standard peak latency:', pre_pert_peak_latency_S)
print('Pre-perturbation Deviant peak latency:', pre_pert_peak_latency_D)
print(" ")

neg_w_perturb_peak_latency_S = np.zeros([8,], dtype ='object')
neg_w_perturb_peak_latency_D = np.zeros([8,], dtype = 'object')

for level in range(8):
  pert_level_peak_latency_S = (np.argmax(avg_output_S[level, :])-1)*delta_t
  neg_w_perturb_peak_latency_S[level] = pert_level_peak_latency_S
  pert_level_peak_latency_D = (np.argmax(avg_output_D[level, :])-1)*delta_t
  neg_w_perturb_peak_latency_D[level] = pert_level_peak_latency_D

print('Post-perturbation Standard peak latencies:', neg_w_perturb_peak_latency_S)
print('Post-perturbation Deviant peak latencies:', neg_w_perturb_peak_latency_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_latency_S.npy', neg_w_perturb_peak_latency_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_peak_latency_D.npy', neg_w_perturb_peak_latency_D)



In [ ]:
""" Plot relative changes in peak amplitude and peak latency from pre-perturb to post-perturb."""

pert_vec = [0, 0.5/100, 1/100, 1.5/100, 2/100, 2.5/100, 3/100, 3.5/100, 4/100]
x_labels = [f'{x*100:.1f}%' for x in pert_vec]

ratio_peaks_S = [1, 1.13, 1.28, 1.45, 1.64, 1.84, 2.09, 2.38, 2.71]
ratio_peaks_D = [1, 1.12, 1.24, 1.37, 1.5, 1.63, 1.77, 1.93, 2.1]
peak_lat_S = [140, 144, 144, 144, 144, 144, 148, 148, 148]
peak_lat_D = [140, 144, 144, 144, 144, 144, 148, 148, 148]
mmn = [0.09, 0.11, 0.12, 0.14, 0.15, 0.16, 0.18, 0.21, 0.3]
mmn_ratio = [1, 1.22, 1.33, 1.55, 1.66, 1.77, 2, 2.33, 3.33]

plt.figure(figsize=(9.5, 6))

plt.plot(pert_vec, ratio_peaks_S, linestyle = "-", color = 'blue', label = 'Peak amplitude S', linewidth=3)
plt.plot(pert_vec, ratio_peaks_D, linestyle = "--", color = 'orange', label = 'Peak amplitude D', linewidth=3)
plt.plot(pert_vec, mmn_ratio, color = 'green', linestyle = "-", label = 'Mismatch negativity', linewidth=3)
plt.xticks(pert_vec, x_labels, fontsize = 17)
plt.yticks(fontsize = 17)
#plt.plot(pert_vec, peak_lat_S, color = 'gray', linestyle = "-")
#plt.plot(pert_vec, peak_lat_S, color = 'gray', linestyle = "--")
plt.xlabel('Perturbation level', fontsize = 18)
plt.ylabel('Post- : Pre-perturbation ratio', fontsize = 18)

# Remove only the top and right borders (spines)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.legend(loc='upper left', fontsize = 15)
#plt.title('AEF changes from baseline for when reducing inhibition', fontsize = 15)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_post_perturb_increase.pdf')
plt.show()

In [ ]:
## Plot phase diagram for the test array (S + D)
test_array = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data/test_array.npy', allow_pickle = False)

# Relative % changes in weights' absolute value
weight_upd_list = list(np.arange(0.005, 0.04, 0.005))
for el in range(len(weight_upd_list)):
  weight_upd_list[el] = round(weight_upd_list[el], 3)

activations_list_perturb = []

for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_neg_w{weight_upd_list[j]}.h5")
  layer = perturbed_model.layers[3]

  if isinstance(layer, tf.keras.layers.SimpleRNN):
    output = layer.output
    activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)

    activations_list = []
    for input in test_array:
      activations = activation_model.predict(input.reshape(1, 138, 63))
      activations_list.append(activations)

  activations_list_perturb.append(activations_list)


final_activ_list = []

for i in range(len(activations_list_perturb)):
  activations_list = activations_list_perturb[i]
  # Convert activations to numpy array
  activations_array = np.array(activations_list)
  # Get average activation over the 360 trials
  avg_activations_array = np.mean(activations_array, axis = 0)
  # Reshape activations for PCA
  reshaped_avg_activations = avg_activations_array.reshape(-1, 64) # reshape so that array has 64 columns.
  final_activ_list.append(reshaped_avg_activations)
  
  
post_pca_activ_list = []
for i in range(len(final_activ_list)):
  # Perform PCA
  pca = PCA(n_components=2)
  reduced_activations = pca.fit_transform(final_activ_list[i])

  post_pca_activ_list.append(reduced_activations)
  
# Save activations array S & D.
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_PCA_L4_activations.npy', np.array(post_pca_activ_list))


# Make plots for latent RNN activity.
from matplotlib.colors import ListedColormap

post_pca_activ_list = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_PCA_L4_activations.npy', allow_pickle=True)

fig, axs = plt.subplots(2, 4, figsize=(20, 8)) # sharex = True
fig.suptitle(f'Phase diagrams of 4th hidden layer activations for negative weight increase by...', fontsize =20)

for k in range(4):
    legend_handles = []
    for f in range(1, 139):
      line = axs[k//4, k%4].plot(post_pca_activ_list[k%4][f - 1:f + 1, 0], post_pca_activ_list[k%4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      line = axs[k//4 + 1, k%4].plot(post_pca_activ_list[k%4+4][f - 1:f + 1, 0], post_pca_activ_list[k%4+4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      legend_handles.append(line)

    axs[k//4, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4, k%4].tick_params(axis="y", labelsize=16)
    axs[k//4, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4 + 1, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4 + 1, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4 + 1, k%4].tick_params(axis="y", labelsize=16)
    axs[k//4, k%4].set_ylim([-0.6, 0.65])
    axs[k//4, k%4].set_xlim([-1, 1.5])
    axs[k//4 + 1, k%4].set_ylim([-0.6, 0.65])
    axs[k//4 + 1, k%4].set_xlim([-1, 1.5])

    axs[k//4, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k]*100)}%', fontsize = 18)
    axs[k//4 + 1, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k+4]*100)}%', fontsize = 18)

timesteps = np.arange(138)  # 0 to 137

plt.legend(handles=legend_handles, labels=[f'Timestep {f}' for f in range(138 - 1)],
           loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

cmap = ListedColormap(plt.cm.viridis(np.arange(0, 138, 1)))

cax = plt.axes([1, 0.15, 0.02, 0.7])  # [x, y, width, height]
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap), cax=cax, label='Timestep')

fig.tight_layout()
fig.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/neg_w_perturb_PCA_L4_activations.pdf')
plt.show()


#### Experiment 2: Increase of excitatory weights. (1st control experiment)



In [ ]:
# Perturb positive weights (increase by %...) ---> later: do small variations in the level of perturbations between the hidden layers.
simple_rnn_model_w = []

no_layers = 4

new_weight_array_rnn = np.zeros([len(weight_upd_list), no_layers, 2], dtype = 'object')


for j in range(len(weight_upd_list)):
  # Load original model
  simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

  # Perturb hidden layers - L1-L4
  for l in range(no_layers):
    # Get the weights of a particular layer
    simple_rnn_layer_w = simple_rnn_model_orig.layers[l].get_weights()
    # Perturb recurrent weights -> increase positive weights
    simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] > 0)] = simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] > 0)] + np.abs(weight_upd_list[j]*simple_rnn_layer_w[1][np.where(simple_rnn_layer_w[1] > 0)])
    new_weight_array_rnn[j, l, 0] = simple_rnn_layer_w[0] # feedf weight = same
    new_weight_array_rnn[j, l, 1] = simple_rnn_layer_w[1] # recurrent weight = the one above

    # update weights
    simple_rnn_model_orig.layers[l].set_weights([new_weight_array_rnn[j, l, 0], new_weight_array_rnn[j, l, 1], simple_rnn_layer_w[2]])

  simple_rnn_model_w.append(simple_rnn_model_orig) # after the 4th layer has been perturbed, add whole perturbed model to the list.


# Evaluate the models
scores_pos_w_perturb_magn = np.zeros([len(weight_upd_list)], dtype = 'object')

for j in range(len(weight_upd_list)):
  # evaluate the model
  score=simple_rnn_model_w[j].evaluate(test_tf_dataset, verbose=2) # the updated version of the original model
  scores_pos_w_perturb_magn[j] = score

# Save loss and mean sq error for TEST
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/scores_pos_w_perturb_magn.npy', scores_pos_w_perturb_magn)

# Save pertubed models
for j in range(len(weight_upd_list)):
  save_model(simple_rnn_model_w[j], f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_pos_w{weight_upd_list[j]}.h5")


In [ ]:
# Calculate average excitatory / inhibition ratio at the synapse for each layer.

for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_pos_w{weight_upd_list[j]}.h5")

  print(f"Perturb level: {weight_upd_list[j]}")
  print(" ")

  for l in range(no_layers):
    simple_rnn_layer_w = perturbed_model.layers[l].get_weights()
    recurrent_weights_l = simple_rnn_layer_w[1] # choose recurrent weights

    weights_array_flat = recurrent_weights_l.flatten()

    pos_weights_total = weights_array_flat[np.where(weights_array_flat>0)]
    no_pos_weights_total = pos_weights_total.shape[0]
    print(f"Total pos weights layer {1+l}:", no_pos_weights_total)
    neg_weights_total = weights_array_flat[np.where(weights_array_flat<0)]
    no_neg_weights_total = neg_weights_total.shape[0]
    print(f"Total neg weights layer {1+l}:", no_neg_weights_total)

    pos_weights_strength = np.sum(pos_weights_total, axis = 0)
    neg_weights_strength = np.sum(neg_weights_total, axis = 0)

    E_I_strength_ratio = pos_weights_strength/neg_weights_strength
    #print(np.abs(E_I_strength_ratio))
    #print(np.abs(E_I_strength_ratio))

    # Calculate average excitation / synapse
    avg_exc_syn = pos_weights_strength/no_pos_weights_total

    # Calculate average inhibition / synapse
    avg_inh_syn = neg_weights_strength/no_neg_weights_total

    synapse_E_I_strength_ratio = np.abs(avg_exc_syn/avg_inh_syn)
    print(f"Synapse E/I ratio layer {1+l}:", synapse_E_I_strength_ratio)

In [ ]:
""" Run Experiment 2 - perturbed model for S and D inputs."""

output_list_S_250 = []
output_list_S_500 = []
output_list_S_1000 = []

output_list_S_array = np.zeros([3, 8, 138], dtype='object')

output_list_D_250_1000 = []
output_list_D_500_1000 = []
output_list_D_250_500 = [] # other combinations do not differ from these.

output_list_D_array = np.zeros([3, 8, 138], dtype='object')

for j in range(len(weight_upd_list)):
    #perturbed_model = simple_rnn_model_w_l[j][l]
    perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_pos_w{weight_upd_list[j]}.h5")
    layer = perturbed_model.layers[4]

    if isinstance(layer, tf.keras.layers.SimpleRNN):
        output = layer.output
        activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)
        activations_S1 = activation_model.predict(test_array[1].reshape(1, 138, 63)) # 250 S
        activations_S2 = activation_model.predict(test_array[5].reshape(1, 138, 63)) # 500 S
        activations_S3 = activation_model.predict(test_array[10].reshape(1, 138, 63)) # 1000S
        output_list_S_250.append(activations_S1)
        output_list_S_500.append(activations_S2)
        output_list_S_1000.append(activations_S3)

        activations_D1 = activation_model.predict(test_array[297].reshape(1, 138, 63)) #250 - 1000 D
        output_list_D_250_1000.append(activations_D1)
        activations_D2 = activation_model.predict(test_array[267].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_500_1000.append(activations_D2)
        activations_D3 = activation_model.predict(test_array[295].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_250_500.append(activations_D3)

output_list_S_array[0, :, :] = np.array(output_list_S_250)[:, 0, :, 0]
output_list_S_array[1, :, :] = np.array(output_list_S_500)[:, 0, :, 0]
output_list_S_array[2, :, :] = np.array(output_list_S_1000)[:, 0, :, 0]

output_list_D_array[0, :, :] = np.array(output_list_D_250_1000)[:, 0, :, 0]
output_list_D_array[1, :, :] = np.array(output_list_D_500_1000)[:, 0, :, 0]
output_list_D_array[2, :, :] = np.array(output_list_D_250_500)[:, 0, :, 0]

print(output_list_S_array.shape)
print(output_list_D_array.shape)

# Take average over all 3 S input types (from test set)
avg_output_S = np.mean(output_list_S_array, axis = 0)
avg_output_D = np.mean(output_list_D_array, axis = 0)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_S.npy', avg_output_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_D.npy', avg_output_D)



""" Plot output of control experiment 1."""
plt.figure(figsize=(9.5, 6))
colors=['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta']

avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

control_1_avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_S.npy', allow_pickle=True)
control_1_avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_D.npy', allow_pickle=True)

t = np.arange(0, 137*4+1, 4)

legend = ['0.5%', '1%', '1.5%', '2%', '2.5%', '3%', '3.5%', '4%']

line1, = plt.plot(t, avg_output_S_orig_model, color = 'black', linestyle = "-", label = '0%')
line2, = plt.plot(t, avg_output_D_orig_model, color = 'black', linestyle = "--")
for i in range(8):
  line3, = plt.plot(t, control_1_avg_output_S[i], color = colors[i], linestyle = "-", label = legend[i])
  line4, = plt.plot(t, control_1_avg_output_D[i], color = colors[i], linestyle = "--")
plt.legend(loc = 'upper left', title = "Perturbation level")
plt.xticks(fontsize = 16)
plt.yticks(fontsize = 16)
plt.xlabel('Time (ms)', fontsize = 17)
plt.ylabel('RNN Prediction', fontsize = 17)

plt.ylim(-1, 20)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/pos_w_perturb_predictions_single_plot.pdf')
plt.show()

In [ ]:
''' 1. Max amplitude ratios & MAE MMMN'''

# Pre-pert max amplitudes.
avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

pre_pert_peak_S = np.max(avg_output_S_orig_model)
pre_pert_peak_D = np.max(avg_output_D_orig_model)

pre_pert_MAE = mean_absolute_error(avg_output_S_orig_model, avg_output_D_orig_model)
print("pre_pert_MAE:", pre_pert_MAE)

control_1_peak_ampl_ratios_S = np.zeros([8,], dtype ='object')
control_1_peak_ampl_ratios_D = np.zeros([8,], dtype = 'object')

control_1_peak_ampl_S = np.zeros([8,], dtype ='object')
control_1_peak_ampl_D = np.zeros([8,], dtype = 'object')

control_1_MMN_MAE = np.zeros([8,], dtype ='object')

avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_S.npy', allow_pickle=True)
avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_avg_output_D.npy', allow_pickle=True)

for level in range(8):
  pert_level_peak_S = np.max(avg_output_S[level, :])
  pert_level_peak_D = np.max(avg_output_D[level, :])

  control_1_peak_ampl_S[level] = pert_level_peak_S
  control_1_peak_ampl_D[level] = pert_level_peak_D

  ratio_S_level = pert_level_peak_S/pre_pert_peak_S
  ratio_D_level = pert_level_peak_D/pre_pert_peak_D

  MMN_MAE = mean_absolute_error(avg_output_S[level, :], avg_output_D[level, :])
  control_1_MMN_MAE[level] = MMN_MAE

  control_1_peak_ampl_ratios_S[level] = ratio_S_level
  control_1_peak_ampl_ratios_D[level] = ratio_D_level

print('Standard ratios:', control_1_peak_ampl_ratios_S)
print('Deviant ratios:', control_1_peak_ampl_ratios_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_ampl_ratios_S.npy', control_1_peak_ampl_ratios_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_ampl_ratios_D.npy', control_1_peak_ampl_ratios_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_ampl_S.npy', control_1_peak_ampl_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_ampl_D.npy', control_1_peak_ampl_D)
print('Peak ampl S:', control_1_peak_ampl_S)
print('Peak ampl D:', control_1_peak_ampl_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_MMN_MAE.npy', control_1_MMN_MAE)
print("MAEs:", control_1_MMN_MAE)
print(" ")


''' 2. Peak latency'''

delta_t = 4 # 4ms.

pre_pert_peak_latency_S = (np.argmax(avg_output_S_orig_model)-1)*delta_t # transform in ms. 37th timepoint => 36 intervals => 36 x 4ms = 144ms.
pre_pert_peak_latency_D = (np.argmax(avg_output_D_orig_model)-1)*delta_t

print('Pre-perturbation Standard peak latency:', pre_pert_peak_latency_S)
print('Pre-perturbation Deviant peak latency:', pre_pert_peak_latency_D)
print(" ")

control_1_peak_latency_S = np.zeros([8,], dtype ='object')
control_1_peak_latency_D = np.zeros([8,], dtype = 'object')

for level in range(8):
  pert_level_peak_latency_S = (np.argmax(avg_output_S[level, :])-1)*delta_t
  control_1_peak_latency_S[level] = pert_level_peak_latency_S
  pert_level_peak_latency_D = (np.argmax(avg_output_D[level, :])-1)*delta_t
  control_1_peak_latency_D[level] = pert_level_peak_latency_D

print('Post-perturbation Standard peak latencies:', control_1_peak_latency_S)
print('Post-perturbation Deviant peak latencies:', control_1_peak_latency_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_latency_S.npy', control_1_peak_latency_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_peak_latency_D.npy', control_1_peak_latency_D)


# Make the plot
pert_vec = [0, 0.5/100, 1/100, 1.5/100, 2/100, 2.5/100, 3/100, 3.5/100, 4/100]
x_labels = [f'{x*100:.1f}%' for x in pert_vec]

ratio_peaks_S = [pre_pert_peak_S] + control_1_peak_ampl_ratios_S.tolist()
ratio_peaks_D = [pre_pert_peak_D] + control_1_peak_ampl_ratios_D.tolist()
#peak_lat_S = [140, 144, 144, 144, 144, 144, 148, 148, 148]
#peak_lat_D = [140, 144, 144, 144, 144, 144, 148, 148, 148]
mmn = control_1_MMN_MAE
mmn_ratio = mmn/pre_pert_MAE
mmn_ratio = [1] + mmn_ratio.tolist()

plt.figure(figsize=(9.5, 6))

plt.plot(pert_vec, ratio_peaks_S, linestyle = "-", color = 'blue', label = 'Peak amplitude S', linewidth=3)
plt.plot(pert_vec, ratio_peaks_D, linestyle = "--", color = 'orange', label = 'Peak amplitude D', linewidth=3)
plt.plot(pert_vec, mmn_ratio, color = 'green', linestyle = "-", label = 'Mismatch negativity', linewidth=3)
plt.xticks(pert_vec, x_labels, fontsize = 17)
plt.yticks(fontsize = 17)
plt.xlabel('Perturbation level', fontsize = 18)
plt.ylabel('Post- : Pre-perturbation ratio', fontsize = 18)

# Remove only the top and right borders (spines)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.legend(fontsize=15)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_1_post_perturb_increase.pdf')
plt.show()



In [ ]:
## Plot phase diagram for the test array (S + D)
test_array = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data/test_array.npy', allow_pickle = False)

# Relative % changes in weights' absolute value
weight_upd_list = list(np.arange(0.005, 0.04, 0.005))
for el in range(len(weight_upd_list)):
  weight_upd_list[el] = round(weight_upd_list[el], 3)

activations_list_perturb = []

for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_perturb_pos_w{weight_upd_list[j]}.h5")
  layer = perturbed_model.layers[3]

  if isinstance(layer, tf.keras.layers.SimpleRNN):
    output = layer.output
    activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)

    activations_list = []
    for input in test_array:
      activations = activation_model.predict(input.reshape(1, 138, 63))
      activations_list.append(activations)

  activations_list_perturb.append(activations_list)


final_activ_list = []

for i in range(len(activations_list_perturb)):
  activations_list = activations_list_perturb[i]

  # Convert activations to numpy array
  activations_array = np.array(activations_list)

  # Get average activation over the 360 trials
  avg_activations_array = np.mean(activations_array, axis = 0)

  # Reshape activations for PCA
  reshaped_avg_activations = avg_activations_array.reshape(-1, 64) # reshape so that array has 64 columns.

  final_activ_list.append(reshaped_avg_activations)


post_pca_activ_list = []

for i in range(len(final_activ_list)):

  # Perform PCA
  pca = PCA(n_components=2)
  reduced_activations = pca.fit_transform(final_activ_list[i])

  post_pca_activ_list.append(reduced_activations)
  

# Save activations array S & D.
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/pos_w_perturb_PCA_L4_activations.npy', np.array(post_pca_activ_list))

# Plot PCA phase diagrams.
post_pca_activ_list = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/pos_w_perturb_PCA_L4_activations.npy')

# Make plot
from matplotlib.colors import ListedColormap

fig, axs = plt.subplots(2, 4, figsize=(20, 8)) # sharex = True
fig.suptitle(f'Phase diagrams of 4th hidden layer activations for positive weight increase by...', fontsize =20)

for k in range(4):
    legend_handles = []
    for f in range(1, 139):
      line = axs[k//4, k%4].plot(post_pca_activ_list[k%4][f - 1:f + 1, 0], post_pca_activ_list[k%4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      line = axs[k//4 + 1, k%4].plot(post_pca_activ_list[k%4+4][f - 1:f + 1, 0], post_pca_activ_list[k%4+4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      legend_handles.append(line)

    axs[k//4, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4, k%4].tick_params(axis="y", labelsize=16)
    axs[k//4, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4 + 1, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4 + 1, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4 + 1, k%4].tick_params(axis="y", labelsize=16)

    axs[k//4, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k]*100)}%', fontsize = 18)
    axs[k//4 + 1, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k+4]*100)}%', fontsize = 18)

timesteps = np.arange(138)  # 0 to 137

plt.legend(handles=legend_handles, labels=[f'Timestep {f}' for f in range(138 - 1)],
           loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

cmap = ListedColormap(plt.cm.viridis(np.arange(0, 138, 1)))

cax = plt.axes([1, 0.15, 0.02, 0.7])  # [x, y, width, height]
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap), cax=cax, label='Timestep')

fig.tight_layout()
fig.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/pos_w_perturb_PCA_L4_activations.pdf')
plt.show()



#### Experiment 3: Decrease of inhibitory weights & increase of excitatory weights (2nd control experiment)

In [ ]:
# Perturb a random set of negative + positive weights (increase by %...) ---> later: do small variations in the level of perturbations between the hidden layers.

# Create a 2 x 2 array with random 1 and 0s -> apply this to a recurrent weight vector
np.random.seed(1)
random_targets_1 = np.random.randint(2, size=(64,64)) # size = shape of weights - recurrent weights of layer 2, 3 and 4.

simple_rnn_model_w = []

no_layers = 4

new_weight_array_rnn = np.zeros([len(weight_upd_list), no_layers, 2], dtype = 'object')

for j in range(len(weight_upd_list)):
  # Load original model
  simple_rnn_model_orig = load_model('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_orig.h5')

  # Perturb hidden layers - L1-L4
  for l in range(no_layers):
    # Get the weights of a particular layer
    simple_rnn_layer_w = simple_rnn_model_orig.layers[l].get_weights()

    # multiply the weight matrix with the random_targets array
    modif_weight_m = random_targets_1*simple_rnn_layer_w[1] # recurrent weights

    # increase negative weights
    modif_weight_m[np.where(modif_weight_m < 0)] = modif_weight_m[np.where(modif_weight_m < 0)] + np.abs(weight_upd_list[j]*modif_weight_m[np.where(modif_weight_m < 0)])

    # increase positive weights
    modif_weight_m[np.where(modif_weight_m > 0)] = modif_weight_m[np.where(modif_weight_m > 0)] + np.abs(weight_upd_list[j]*modif_weight_m[np.where(modif_weight_m > 0)])
    new_weight_array_rnn[j, l, 0] = simple_rnn_model_orig.layers[l].get_weights()[0] # ff weights remain the same as in original model.
    new_weight_array_rnn[j, l, 1] = modif_weight_m

    # update weights
    simple_rnn_model_orig.layers[l].set_weights([new_weight_array_rnn[j, l, 0], new_weight_array_rnn[j, l, 1], simple_rnn_layer_w[2]])

  simple_rnn_model_w.append(simple_rnn_model_orig) # after the 4th layer has been perturbed, add whole perturbed model to the list.


# Evaluate the models
scores_control_2_magn = np.zeros([len(weight_upd_list)], dtype = 'object')

for j in range(len(weight_upd_list)):
  # evaluate the model
  score=simple_rnn_model_w[j].evaluate(test_tf_dataset, verbose=2) # the updated version of the original model
  scores_control_2_magn[j] = score

# Save loss and mean sq error for TEST
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/scores_control_2_magn.npy', scores_control_2_magn)

# Save pertubed models
for j in range(len(weight_upd_list)):
  save_model(simple_rnn_model_w[j], f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_control_2{weight_upd_list[j]}.h5")


In [ ]:
# Check E/I strength ratio at synapse for each layer.
for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_control_2{weight_upd_list[j]}.h5")

  print(f"Perturb level: {weight_upd_list[j]}")

  for l in range(no_layers):
    simple_rnn_layer_w = perturbed_model.layers[l].get_weights()
    recurrent_weights_l = simple_rnn_layer_w[1] # choose recurrent weights

    weights_array_flat = recurrent_weights_l.flatten()

    pos_weights_total = weights_array_flat[np.where(weights_array_flat>0)]
    no_pos_weights_total = pos_weights_total.shape[0]
    print(f"Total pos weights layer {1+l}:", no_pos_weights_total)
    neg_weights_total = weights_array_flat[np.where(weights_array_flat<0)]
    no_neg_weights_total = neg_weights_total.shape[0]
    print(f"Total neg weights layer {1+l}:", no_neg_weights_total)

    pos_weights_strength = np.sum(pos_weights_total, axis = 0)
    neg_weights_strength = np.sum(neg_weights_total, axis = 0)

    E_I_strength_ratio = pos_weights_strength/neg_weights_strength

    # Calculate average excitation / synapse
    avg_exc_syn = pos_weights_strength/no_pos_weights_total

    # Calculate average inhibition / synapse
    avg_inh_syn = neg_weights_strength/no_neg_weights_total

    synapse_E_I_strength_ratio = np.abs(avg_exc_syn/avg_inh_syn)
    print(f"Synapse E/I ratio layer {1+l}:", synapse_E_I_strength_ratio)

In [ ]:
""" Run Experiment 3 - perturbed model for S and D inputs."""

output_list_S_250 = []
output_list_S_500 = []
output_list_S_1000 = []

output_list_S_array = np.zeros([3, 8, 138], dtype='object')

output_list_D_250_1000 = []
output_list_D_500_1000 = []
output_list_D_250_500 = [] # other combinations do not differ from these.

output_list_D_array = np.zeros([3, 8, 138], dtype='object')

for j in range(len(weight_upd_list)):
    #perturbed_model = simple_rnn_model_w_l[j][l]
    perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_control_2{weight_upd_list[j]}.h5")
    layer = perturbed_model.layers[4]

    if isinstance(layer, tf.keras.layers.SimpleRNN):
        output = layer.output
        activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)
        activations_S1 = activation_model.predict(test_array[1].reshape(1, 138, 63)) # 250 S
        activations_S2 = activation_model.predict(test_array[5].reshape(1, 138, 63)) # 500 S
        activations_S3 = activation_model.predict(test_array[10].reshape(1, 138, 63)) # 1000S
        output_list_S_250.append(activations_S1)
        output_list_S_500.append(activations_S2)
        output_list_S_1000.append(activations_S3)

        activations_D1 = activation_model.predict(test_array[297].reshape(1, 138, 63)) #250 - 1000 D
        output_list_D_250_1000.append(activations_D1)
        activations_D2 = activation_model.predict(test_array[267].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_500_1000.append(activations_D2)
        activations_D3 = activation_model.predict(test_array[295].reshape(1, 138, 63)) #500 - 1000 D
        output_list_D_250_500.append(activations_D3)

output_list_S_array[0, :, :] = np.array(output_list_S_250)[:, 0, :, 0]
output_list_S_array[1, :, :] = np.array(output_list_S_500)[:, 0, :, 0]
output_list_S_array[2, :, :] = np.array(output_list_S_1000)[:, 0, :, 0]

output_list_D_array[0, :, :] = np.array(output_list_D_250_1000)[:, 0, :, 0]
output_list_D_array[1, :, :] = np.array(output_list_D_500_1000)[:, 0, :, 0]
output_list_D_array[2, :, :] = np.array(output_list_D_250_500)[:, 0, :, 0]

print(output_list_S_array.shape)
print(output_list_D_array.shape)

# Take average over all 3 S input types (from test set)
avg_output_S = np.mean(output_list_S_array, axis = 0)
avg_output_D = np.mean(output_list_D_array, axis = 0)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_S.npy', avg_output_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_D.npy', avg_output_D)


""" Plot predictions """
plt.figure(figsize=(9.5, 6))

colors=['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta']

avg_output_S_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_S_orig_model.npy', allow_pickle=True)
avg_output_D_orig_model = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/avg_output_D_orig_model.npy', allow_pickle=True)

control_2_avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_S.npy', allow_pickle=True)
control_2_avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_D.npy', allow_pickle=True)

t = np.arange(0, 137*4+1, 4)

legend = ['0.5%', '1%', '1.5%', '2%', '2.5%', '3%', '3.5%', '4%']

line1, = plt.plot(t, avg_output_S_orig_model, color = 'black', linestyle = "-", label = '0%')
line2, = plt.plot(t, avg_output_D_orig_model, color = 'black', linestyle = "--")
for i in range(8):
  line3, = plt.plot(t, control_2_avg_output_S[i], color = colors[i], linestyle = "-", label = legend[i])
  line4, = plt.plot(t, control_2_avg_output_D[i], color = colors[i], linestyle = "--")
plt.xticks(fontsize = 17)
plt.yticks(fontsize = 17)
plt.xlabel('Time (ms)', fontsize = 18)
plt.ylabel('RNN Prediction', fontsize = 18)

# Remove only the top and right borders (spines)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

#plt.ylim(-1, 20)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_w_perturb_predictions_single_plot.pdf')
plt.show()


In [ ]:
# Plot loss.

scores_control_2_magn = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/scores_control_2_magn.npy', allow_pickle=True)
# this is an array with 8 lists: in each list, list[0] = loss; list[1] = MSE. they are the same.

data = {f'{round(weight_upd_list[0]*100, 2)}%': scores_control_2_magn[0][0], f'{round(weight_upd_list[1]*100, 2)}%': scores_control_2_magn[1][0], f'{round(weight_upd_list[2]*100,2)}%': scores_control_2_magn[2][0],
        f'{round(weight_upd_list[3]*100, 2)}%': scores_control_2_magn[3][0], f'{round(weight_upd_list[4]*100, 2)}%': scores_control_2_magn[4][0], f'{round(weight_upd_list[5]*100, 2)}%': scores_control_2_magn[5][0],
        f'{round(weight_upd_list[6]*100, 2)}%': scores_control_2_magn[6][0], f'{round(weight_upd_list[7]*100, 2)}%': scores_control_2_magn[7][0]}

plt.bar(data.keys(), data.values(),  color=['rosybrown', 'indianred', 'coral', 'saddlebrown', 'cornflowerblue', 'slateblue', 'darkviolet', 'magenta'])
plt.tick_params(labelsize=12)
plt.ylabel('loss', fontsize = 14)
plt.xlabel('Relative increase of a random set of weights', fontsize = 14)
plt.title('Loss across perturbation levels', fontsize = 18)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/loss_control_2.pdf')
plt.show()

In [ ]:
''' 1. Max amplitude ratios'''

pos_w_perturb_peak_ampl_ratios_S = np.zeros([8,], dtype ='object')
pos_w_perturb_peak_ampl_ratios_D = np.zeros([8,], dtype = 'object')

avg_output_S = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_S.npy', allow_pickle=True)
avg_output_D = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_avg_output_D.npy', allow_pickle=True)

pre_pert_peak_S = 0.94
pre_pert_peak_D = 1.37

for level in range(8):
  pert_level_peak_S = np.max(avg_output_S[level, :])
  pert_level_peak_D = np.max(avg_output_D[level, :])

  ratio_S_level = pert_level_peak_S/pre_pert_peak_S
  ratio_D_level = pert_level_peak_D/pre_pert_peak_D

  pos_w_perturb_peak_ampl_ratios_S[level] = ratio_S_level
  pos_w_perturb_peak_ampl_ratios_D[level] = ratio_D_level

print('Standard ratios:', pos_w_perturb_peak_ampl_ratios_S)
print('Deviant ratios:', pos_w_perturb_peak_ampl_ratios_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_peak_ampl_ratios_S.npy', pos_w_perturb_peak_ampl_ratios_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_peak_ampl_ratios_D.npy', pos_w_perturb_peak_ampl_ratios_D)


''' 2. Peak latency'''

delta_t = 4 # 4ms.

pos_w_perturb_peak_latency_S = np.zeros([8,], dtype ='object')
pos_w_perturb_peak_latency_D = np.zeros([8,], dtype = 'object')

for level in range(8):
  pert_level_peak_latency_S = (np.argmax(avg_output_S[level, :])-1)*delta_t
  pos_w_perturb_peak_latency_S[level] = pert_level_peak_latency_S
  pert_level_peak_latency_D = (np.argmax(avg_output_D[level, :])-1)*delta_t
  pos_w_perturb_peak_latency_D[level] = pert_level_peak_latency_D

print('Post-perturbation Standard peak latencies:', pos_w_perturb_peak_latency_S)
print('Post-perturbation Deviant peak latencies:', pos_w_perturb_peak_latency_D)

np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_peak_latency_S.npy', pos_w_perturb_peak_latency_S)
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_peak_latency_D.npy', pos_w_perturb_peak_latency_D)

In [ ]:
## Plot phase diagram for the test array (S + D)

test_array = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data/test_array.npy', allow_pickle = False)

# Relative % changes in weights' absolute value
weight_upd_list = list(np.arange(0.005, 0.04, 0.005))
for el in range(len(weight_upd_list)):
  weight_upd_list[el] = round(weight_upd_list[el], 3)

activations_list_perturb = []

for j in range(len(weight_upd_list)):
  perturbed_model = load_model(f"/content/drive/My Drive/S_D_ERP_RNN_final/data_final/simple_rnn_model_control_2{weight_upd_list[j]}.h5")
  layer = perturbed_model.layers[3]

  if isinstance(layer, tf.keras.layers.SimpleRNN):
    output = layer.output
    activation_model = tf.keras.models.Model(inputs=perturbed_model.input, outputs=output)

    activations_list = []
    for input in test_array:
      activations = activation_model.predict(input.reshape(1, 138, 63))
      activations_list.append(activations)

  activations_list_perturb.append(activations_list)


final_activ_list = []

for i in range(len(activations_list_perturb)):
  activations_list = activations_list_perturb[i]
  # Convert activations to numpy array
  activations_array = np.array(activations_list)
  # Get average activation over the 360 trials
  avg_activations_array = np.mean(activations_array, axis = 0)
  # Reshape activations for PCA
  reshaped_avg_activations = avg_activations_array.reshape(-1, 64) # reshape so that array has 64 columns.
  final_activ_list.append(reshaped_avg_activations)

post_pca_activ_list = []

for i in range(len(final_activ_list)):
  # Perform PCA
  pca = PCA(n_components=2)
  reduced_activations = pca.fit_transform(final_activ_list[i])
  post_pca_activ_list.append(reduced_activations)
  
# Save activations array S & D.
np.save('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_PCA_L4_activations.npy', np.array(post_pca_activ_list))

In [ ]:
# Plot PCA phase diagrams.

post_pca_activ_list = np.load('/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_PCA_L4_activations.npy', allow_pickle=True)

fig, axs = plt.subplots(2, 4, figsize=(20, 8)) # sharex = True
fig.suptitle(f'Phase diagrams of 4th hidden layer activations for random set of weights increased by...', fontsize =20)

for k in range(4):
    legend_handles = []
    for f in range(1, 139):
      line = axs[k//4, k%4].plot(post_pca_activ_list[k%4][f - 1:f + 1, 0], post_pca_activ_list[k%4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      line = axs[k//4 + 1, k%4].plot(post_pca_activ_list[k%4+4][f - 1:f + 1, 0], post_pca_activ_list[k%4+4][f - 1:f + 1, 1], color=plt.cm.viridis(f / 138), linewidth=2)
      legend_handles.append(line)

    axs[k//4, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4, k%4].tick_params(axis="y", labelsize=16)
    axs[k//4, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].set_xlabel('PC1', fontsize = 17)
    axs[k//4 + 1, k%4].set_ylabel('PC2', fontsize = 17)
    axs[k//4 + 1, k%4].set_title("Phase Diagram of Activations' Trajectories")
    axs[k//4 + 1, k%4].tick_params(axis="x", labelsize=16)
    axs[k//4 + 1, k%4].tick_params(axis="y", labelsize=16)
    #axs[k//4, k%4].set_ylim([-0.6, 0.65])
    #axs[k//4, k%4].set_xlim([-1, 1.5])
    #axs[k//4 + 1, k%4].set_ylim([-0.6, 0.65])
    #axs[k//4 + 1, k%4].set_xlim([-1, 1.5])

    axs[k//4, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k]*100)}%', fontsize = 18)
    axs[k//4 + 1, k%4].set_title(f'{"{:.1f}".format(weight_upd_list[k+4]*100)}%', fontsize = 18)

timesteps = np.arange(138)  # 0 to 137

plt.legend(handles=legend_handles, labels=[f'Timestep {f}' for f in range(138 - 1)],
           loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

cmap = ListedColormap(plt.cm.viridis(np.arange(0, 138, 1)))

cax = plt.axes([1, 0.15, 0.02, 0.7])  # [x, y, width, height]
plt.colorbar(plt.cm.ScalarMappable(cmap=cmap), cax=cax, label='Timestep')

fig.tight_layout()
fig.savefig(f'/content/drive/My Drive/S_D_ERP_RNN_final/data_final/control_2_perturb_PCA_L4_activations.pdf')
plt.show()